# MarketPulse: Multi-Source Data Intelligence Pipeline
### A CRISP-DM Analysis of Price Movements and News Activity

In [1]:
!pip install pandas requests beautifulsoup4

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

print("All libraries imported successfully.")

All libraries imported successfully.


## 1. Business Understanding
Sudden stock price movements are often linked to news events, but the relationship 
isn't always obvious in real time. This project investigates whether spikes in 
news headline volume coincide with significant price changes for a given stock — 
information that could help analysts prioritize which price movements warrant 
closer investigation.

In [3]:
API_KEY = "YOUR_API_KEY_HERE"
SYMBOL = "IBM"  # you can change this to your own stock

In [4]:
import requests

url = "https://www.alphavantage.co/query"

params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "apikey": API_KEY,
    "outputsize": "compact"  # It will give recent 100 days of data
}

response = requests.get(url, params=params)
data = response.json()

print(data.keys())

dict_keys(['Meta Data', 'Time Series (Daily)'])


## 2. Data Understanding
Two independent data sources were used:
- **Price data:** Daily OHLCV (open, high, low, close, volume) data from the 
  Alpha Vantage API, covering the last 100 trading days.
- **News data:** Headline data from Yahoo Finance's public RSS feed, which 
  only covers a limited recent window (typically the last few days to weeks).

This difference in time coverage is an important limitation, addressed directly 
in the Data Preparation step below.

In [9]:
!pip install feedparser

In [10]:
import feedparser

# RSS feed for IBM symbol that we used in Above
rss_url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={SYMBOL}&region=US&lang=en-US"

feed = feedparser.parse(rss_url)

print(f"Number of headlines found: {len(feed.entries)}")

Number of headlines found: 20


## 3. Data Preparation
The two sources were reconciled by extracting a common `date` field from each 
and merging on that field. Days without any news coverage were assigned a 
`headline_count` of 0 rather than being dropped, to preserve the full price 
history for analysis.

In [16]:
# Price data: extract the date (no_time) from index
df_price = df.copy()
df_price["date"] = df_price.index.date

# News data: extract the date (no_time) from timestamp
df_news = news_df.copy()
df_news["date"] = df_news["published"].dt.date

df_price.head()

In [17]:
daily_headline_counts = df_news.groupby("date").size().reset_index(name="headline_count")

daily_headline_counts.head()

In [18]:
merged_df = pd.merge(df_price, daily_headline_counts, on="date", how="left")

# Days with no news all show NaN — replace with 0
merged_df["headline_count"] = merged_df["headline_count"].fillna(0)

merged_df.head(10)

## 4. Modeling (Analytical Layer)
Rather than a machine learning model, a transparent, rule-based approach was 
used: rolling averages and volatility were calculated over a 5-day window, and 
an anomaly was defined as a day where price movement exceeded 3% AND headline 
activity exceeded 2 articles — a deliberately interpretable rule rather than a 
black-box prediction.

In [23]:
merged_df["pct_change"] = merged_df["close"].pct_change() * 100

merged_df[["close", "pct_change"]].head(10)

In [24]:
merged_df["rolling_avg_close"] = merged_df["close"].rolling(window=5).mean()
merged_df["volatility"] = merged_df["close"].rolling(window=5).std()

merged_df[["close", "rolling_avg_close", "volatility"]].head(10)

In [25]:
CHANGE_THRESHOLD = 3.0      # price move more than 3%
HEADLINE_THRESHOLD = 2      # more than 2 headlines a day

merged_df["anomaly_flag"] = (
    (merged_df["pct_change"].abs() > CHANGE_THRESHOLD) &
    (merged_df["headline_count"] > HEADLINE_THRESHOLD)
)

merged_df[merged_df["anomaly_flag"] == True]

In [27]:
print("Anomaly Detection Summary")
print("-" * 30)
print(f"Total days analyzed: {len(merged_df)}")
print(f"Days with anomaly flag = True: {merged_df['anomaly_flag'].sum()}")
print(f"Max single-day price change observed: {merged_df['pct_change'].abs().max():.2f}%")
print(f"Max headline count in a single day: {merged_df['headline_count'].max()}")

Anomaly Detection Summary
------------------------------
Total days analyzed: 100
Days with anomaly flag = True: 0
Max single-day price change observed: 25.21%
Max headline count in a single day: 8.0


## 5. Evaluation
No days met both anomaly conditions within the observed window. The maximum 
single-day price change was 25.21%, and the maximum 
headline count in a single day was 8.0. This suggests 
the stock experienced relatively stable price behavior during this period, 
and/or that the news observation window was too short to capture a major 
news-driven event.

## 6. Recommendations
A longer news collection window (e.g., accumulating headlines daily over 
several months via a scheduled script) would allow this system to be tested 
against a wider range of market conditions, including periods of higher 
volatility. This kind of pipeline could support analysts by automatically 
surfacing days that warrant closer manual review.

In [28]:
import os
print("Files saved in this project:")
for f in ["price_data_raw.csv", "headlines_raw.csv", "merged_data.csv", "final_analysis.csv"]:
    print(f"- {f}: {'✅ found' if os.path.exists(f) else '❌ missing'}")

Files saved in this project:
- price_data_raw.csv: ✅ found
- headlines_raw.csv: ✅ found
- merged_data.csv: ✅ found
- final_analysis.csv: ✅ found
